# 使用函数调用查找附近地点：利用 Google Places API 和客户画像

本 notebook 围绕 Google Places API 与自定义用户画像的集成展开，以增强基于位置的搜索。我们的方法是将 Google Places API 与用户偏好结合使用，旨在让地点发现更加个性化、更具相关性。请注意，虽然本例聚焦于 Google Places API，但你还可以探索许多其他 API，并以类似方式加以应用。

我们将探讨三个主要组成部分的应用：

- 客户画像：这个模拟画像记录了个人对地点类型（例如餐厅、公园、博物馆）、预算、偏好评分以及其他具体要求的偏好。 

- Google Places API：该 API 提供附近地点的实时数据。它会综合考虑你周边地点的评分、场所类型、费用等各种数据点。

- 函数调用：一条诸如“我饿了”或“我想去参观博物馆”的指令即可触发函数，该函数结合用户画像数据和 Google Places API 来找出合适的场所。

本 notebook 介绍两个主要用例：

- 基于画像的推荐：了解如何创建用户画像，并根据个人偏好推荐地点。

- 通过函数调用集成 API：了解如何有效地集成和调用 Google Places API，借助函数调用获取各类地点的实时数据。

请注意，虽然该系统非常灵活，但其效果可能因用户偏好和可用的地点数据而异。在本 notebook 中，客户数据是虚构的，位置也是硬编码的。 

## 设置

Google Places API

要使用 Google Places API，你需要准备两样东西：

- Google 账号：如果还没有，你需要创建一个 Google 账号。

- Google Places API 密钥：API 密钥是一个唯一标识符，用于对与你的项目相关的请求进行身份验证，以便统计用量和计费。你可以从 [Google Cloud Console](https://console.cloud.google.com/getting-started?authuser=1) 获取 API 密钥。 



请注意，Google Places API 是付费服务，费用与 API 调用次数相关。请跟踪你的用量，以免产生意外费用。



还需要 requests 库，你可以使用以下命令下载： 

```python
pip install requests

In [1]:
import json
from openai import OpenAI
import os
import requests

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "<your OpenAI API key if not set as env var>"))

在这段代码中，我们定义了一个函数 `fetch_customer_profile`，它接收 `user_id` 并返回一个模拟的用户画像。

该函数模拟从数据库获取用户数据的 API 调用。在本演示中，我们使用硬编码数据。用户画像包含各种详细信息，例如用户位置（本例中设为金门大桥的坐标）、饮食和活动偏好、应用使用指标、最近的互动以及用户等级。

在生产环境中，你应将这些硬编码数据替换为对用户数据库的真实 API 调用。


In [2]:
def fetch_customer_profile(user_id):
    # You can replace this with a real API call in the production code
    if user_id == "user1234":
        return {
            "name": "John Doe",
            "location": {
                "latitude": 37.7955,
                "longitude": -122.4026,
            },
            "preferences": {
                "food": ["Italian", "Sushi"],
                "activities": ["Hiking", "Reading"],
            },
            "behavioral_metrics": {
                "app_usage": {
                    "daily": 2,  # hours
                    "weekly": 14  # hours
                },
                "favourite_post_categories": ["Nature", "Food", "Books"],
                "active_time": "Evening",
            },
            "recent_searches": ["Italian restaurants nearby", "Book clubs"],
            "recent_interactions": ["Liked a post about 'Best Pizzas in New York'", "Commented on a post about 'Central Park Trails'"],
            "user_rank": "Gold",  # based on some internal ranking system
        }
    else:
        return None


## 从 Google Places API 请求并处理数据

函数 call_google_places_api 用于向 Google Places API 请求信息，并根据给定的 place_type 和可选的 food_preference 提供排名前两位的地点列表。由于这是付费服务，为控制用量，我们将该函数限制为只返回前两个结果。不过，你可以根据需要修改它以获取任意数量的结果。

该函数配置了硬编码的位置（设为泛美金字塔的坐标）、你的 Google API 密钥以及特定的请求参数。它会根据 place_type 构造相应的 API 请求 URL。如果 place_type 是餐厅且指定了 food_preference，则会将其包含在 API 请求中。

发送 GET 请求后，函数会检查响应状态。如果成功，它会处理 JSON 响应，使用 get_place_details 函数提取相关详细信息，并以人类可读的格式返回。如果请求失败，它会打印错误以便调试。

get_place_details 函数用于根据 place_id 获取某个地点的更详细信息。它向 Google Place Details API 发送 GET 请求，如果请求成功则返回结果。如果请求失败，它会打印错误以便调试。

两个函数都会处理异常，并在出现问题时返回错误消息。 

In [3]:
def get_place_details(place_id, api_key):
    URL = f"https://maps.googleapis.com/maps/api/place/details/json?place_id={place_id}&key={api_key}"
    response = requests.get(URL)
    if response.status_code == 200:
        result = json.loads(response.content)["result"]
        return result
    else:
        print(f"Google Place Details API request failed with status code {response.status_code}")
        print(f"Response content: {response.content}")
        return None


In [4]:
def call_google_places_api(user_id, place_type, food_preference=None):
    try:
        # Fetch customer profile
        customer_profile = fetch_customer_profile(user_id)
        if customer_profile is None:
            return "I couldn't find your profile. Could you please verify your user ID?"

        # Get location from customer profile
        lat = customer_profile["location"]["latitude"]
        lng = customer_profile["location"]["longitude"]

        API_KEY = os.getenv('GOOGLE_PLACES_API_KEY')  # retrieve API key from environment variable
        LOCATION = f"{lat},{lng}"
        RADIUS = 500  # search within a radius of 500 meters
        TYPE = place_type

        # If the place_type is restaurant and food_preference is not None, include it in the API request
        if place_type == 'restaurant' and food_preference:
            URL = f"https://maps.googleapis.com/maps/api/place/nearbysearch/json?location={LOCATION}&radius={RADIUS}&type={TYPE}&keyword={food_preference}&key={API_KEY}"
        else:
            URL = f"https://maps.googleapis.com/maps/api/place/nearbysearch/json?location={LOCATION}&radius={RADIUS}&type={TYPE}&key={API_KEY}"

        response = requests.get(URL)
        if response.status_code == 200:
            results = json.loads(response.content)["results"]
            places = []
            for place in results[:2]:  # limit to top 2 results
                place_id = place.get("place_id")
                place_details = get_place_details(place_id, API_KEY)  # Get the details of the place

                place_name = place_details.get("name", "N/A")
                place_types = next((t for t in place_details.get("types", []) if t not in ["food", "point_of_interest"]), "N/A")  # Get the first type of the place, excluding "food" and "point_of_interest"
                place_rating = place_details.get("rating", "N/A")  # Get the rating of the place
                total_ratings = place_details.get("user_ratings_total", "N/A")  # Get the total number of ratings
                place_address = place_details.get("vicinity", "N/A")  # Get the vicinity of the place

                if ',' in place_address:  # If the address contains a comma
                    street_address = place_address.split(',')[0]  # Split by comma and keep only the first part
                else:
                    street_address = place_address

                # Prepare the output string for this place
                place_info = f"{place_name} is a {place_types} located at {street_address}. It has a rating of {place_rating} based on {total_ratings} user reviews."

                places.append(place_info)

            return places
        else:
            print(f"Google Places API request failed with status code {response.status_code}")
            print(f"Response content: {response.content}")  # print out the response content for debugging
            return []
    except Exception as e:
        print(f"Error during the Google Places API call: {e}")
        return []


## 使用 GPT-3.5-Turbo 和 Google Places API 生成用户专属推荐

函数 `provide_user_specific_recommendations` 与 GPT-3.5-Turbo 和 Google Places API 交互，提供根据用户偏好和位置量身定制的回复。

首先，它使用客户的 `user_id` 获取其画像。如果找不到画像，则返回错误消息。

获得有效画像后，它会提取客户的饮食偏好，然后与 OpenAI 模型交互。它会提供一条初始系统消息，向 AI 模型说明其角色、用户偏好以及 Google Places API 函数的用法。

用户输入也会作为消息发送给模型，函数 `call_google_places_api` 则定义在 `functions` 参数中，供 AI 模型按需调用。

最后，它会处理模型的响应。如果模型发起了对 Google Places API 的函数调用，就会使用相应参数执行该函数，并返回附近地点的名称。如果没有找到这样的地点或无法理解请求，则返回相应的错误消息。


In [7]:
def provide_user_specific_recommendations(user_input, user_id):
    customer_profile = fetch_customer_profile(user_id)
    if customer_profile is None:
        return "I couldn't find your profile. Could you please verify your user ID?"

    customer_profile_str = json.dumps(customer_profile)

    food_preference = customer_profile.get('preferences', {}).get('food', [])[0] if customer_profile.get('preferences', {}).get('food') else None


    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
    {
        "role": "system",
        "content": f"You are a sophisticated AI assistant, a specialist in user intent detection and interpretation. Your task is to perceive and respond to the user's needs, even when they're expressed in an indirect or direct manner. You excel in recognizing subtle cues: for example, if a user states they are 'hungry', you should assume they are seeking nearby dining options such as a restaurant or a cafe. If they indicate feeling 'tired', 'weary', or mention a long journey, interpret this as a request for accommodation options like hotels or guest houses. However, remember to navigate the fine line of interpretation and assumption: if a user's intent is unclear or can be interpreted in multiple ways, do not hesitate to politely ask for additional clarification. Make sure to tailor your responses to the user based on their preferences and past experiences which can be found here {customer_profile_str}"
    },
    {"role": "user", "content": user_input}
],
        temperature=0,
        tools=[
            {
                "type": "function",
                "function" : {
                    "name": "call_google_places_api",
                    "description": "This function calls the Google Places API to find the top places of a specified type near a specific location. It can be used when a user expresses a need (e.g., feeling hungry or tired) or wants to find a certain type of place (e.g., restaurant or hotel).",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "place_type": {
                                "type": "string",
                                "description": "The type of place to search for."
                            }
                        }
                    },
                    "result": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    }
                }
            }
        ],
    )

    print(response.choices[0].message.tool_calls)

    if response.choices[0].finish_reason=='tool_calls':
        function_call = response.choices[0].message.tool_calls[0].function
        if function_call.name == "call_google_places_api":
            place_type = json.loads(function_call.arguments)["place_type"]
            places = call_google_places_api(user_id, place_type, food_preference)
            if places:  # If the list of places is not empty
                return f"Here are some places you might be interested in: {' '.join(places)}"
            else:
                return "I couldn't find any places of interest nearby."

    return "I am sorry, but I could not understand your request."



## 执行用户专属推荐

执行时，该函数会获取用户画像，与 AI 模型交互，处理模型的响应，必要时调用 Google Places API，最终返回根据用户偏好和位置量身定制的推荐列表。打印输出将包含这些个性化推荐。

In [8]:
user_id = "user1234"
user_input = "I'm hungry"
output = provide_user_specific_recommendations(user_input, user_id)
print(output)


[ChatCompletionMessageToolCall(id='call_Q1mXIi7D6GhobfE4tkruX7nB', function=Function(arguments='{\n  "place_type": "restaurant"\n}', name='call_google_places_api'), type='function')]
Here are some places you might be interested in: Sotto Mare is a restaurant located at 552 Green Street. It has a rating of 4.6 based on 3765 user reviews. Mona Lisa Restaurant is a restaurant located at 353 Columbus Avenue #3907. It has a rating of 4.4 based on 1888 user reviews.
